# GeoAI Aquaculture Pond Identification — Final Submission

Place `Train.csv`, `Test.csv`, and `SampleSubmission.csv` in `data/raw/`, then install the validated extras with `python -m pip install -e ".[dev,trees,deep,notebook]"`. This notebook only orchestrates tested project modules; it does not duplicate feature, validation, model, ensemble, calibration, or submission logic.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from geoai_aquaculture.constants import FIXED_THRESHOLD
from geoai_aquaculture.data import load_project_config
from geoai_aquaculture.ensemble import (
    build_final_delivery,
    load_final_delivery_config,
)
from geoai_aquaculture.submission import validate_submission

final_config = load_final_delivery_config(Path("configs/final.yaml"))
project = load_project_config(final_config.project_config)
assert final_config.threshold == FIXED_THRESHOLD == 0.5
print(
    {
        "validation_seed": project.validation.seed,
        "folds": project.validation.n_splits,
        "repeats": project.validation.n_repeats,
        "threshold": final_config.threshold,
        "config_fingerprint": final_config.fingerprint,
    }
)

In [ ]:
result = build_final_delivery(final_config, project=project, reuse_existing=True)
result

In [ ]:
submission = pd.read_csv(result.submission_path)
sample = pd.read_csv(project.data.sample_submission_path)
validate_submission(submission, sample)
metrics = json.loads((result.output_dir / "metrics.json").read_text())
models = json.loads((result.output_dir / "full_data_models.json").read_text())
display(submission.head())
display(metrics["selected_oof"])
display(models)

In [ ]:
trustworthiness = (result.output_dir / "trustworthiness.md").read_text()
print(trustworthiness)

## Audit trail

The final directory contains candidate and OOF hashes, nested blend weights, calibration comparisons, model hashes, SHAP outputs, per-row tree/GRU disagreement, prior-shift diagnostics, runtime metadata, trustworthiness responses, and the exact submission hash.